# 허깅페이스에서_모델받아_다국어번역_서비스만들기

In [2]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

ko_text = "이것은 m2m모델로 만든 다국어 번역기 입니다."
chinese_text = "生活就像一盒巧克力。"

model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 한국어를 영어로
tokenizer.src_lang = "ko"
encoded_hi = tokenizer(ko_text , return_tensors="pt")
generated_tokens = model.generate(**encoded_hi, forced_bos_token_id=tokenizer.get_lang_id("en"))
result1 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result1)
# => "La vie est comme une boîte de chocolat."

# translate Chinese to English
tokenizer.src_lang = "ko"
encoded_zh = tokenizer(ko_text, return_tensors="pt")
generated_tokens = model.generate(**encoded_zh, forced_bos_token_id=tokenizer.get_lang_id("ja"))
result2 = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
print(result2)
# => "Life is like a box of chocolate."


['This is a multi-language translator made with the m2m model.']
['これはm2mモデルで作られた多言語翻訳機です。']


In [3]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

# 모델과 토크나이저 로드
model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 지원 언어 목록 (일부만 추림, 전체 언어 사용 시 tokenizer.lang_code_to_id.keys() 사용)
LANGUAGES = {
    "Korean": "ko",
    "English": "en",
    "Chinese": "zh",
    "Japanese": "ja",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Hindi": "hi"
}

def translate_text(text, src_lang_name, tgt_lang_name):
    if not text.strip():
        return ""
    
    src_lang = LANGUAGES[src_lang_name]
    tgt_lang = LANGUAGES[tgt_lang_name]

    tokenizer.src_lang = src_lang
    encoded = tokenizer(text, return_tensors="pt")
    generated_tokens = model.generate(
        **encoded, forced_bos_token_id=tokenizer.get_lang_id(tgt_lang)
    )
    result = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)
    return result[0]

# Gradio UI 구성
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 다국어 번역기 (M2M100 기반)")

    with gr.Row():
        src_lang = gr.Dropdown(list(LANGUAGES.keys()), label="원본 언어", value="Korean")
        tgt_lang = gr.Dropdown(list(LANGUAGES.keys()), label="번역 언어", value="English")

    with gr.Row():
        input_text = gr.Textbox(lines=8, label="원본 텍스트")
        output_text = gr.Textbox(lines=8, label="번역 결과", interactive=False)

    translate_button = gr.Button("번역하기")

    translate_button.click(fn=translate_text, inputs=[input_text, src_lang, tgt_lang], outputs=output_text)

demo.launch()


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [4]:
demo.close()

Closing server running on port: 7860


In [6]:
!pip install datasets soundfile

  Using cached cffi-1.17.1-cp311-cp311-win_amd64.whl.metadata (1.6 kB)
  Using cached pycparser-2.22-py3-none-any.whl.metadata (943 bytes)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 5.4 MB/s eta 0:00:00
Using cached cffi-1.17.1-cp311-cp311-win_amd64.whl (181 kB)
   ---------------------------------------- 0.0/25.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/25.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/25.8 MB 2.8 MB/s eta 0:00:09
   - -------------------------------------- 0.8/25.8 MB 2.8 MB/s eta 0:00:09
   - -------------------------------------- 1.0/25.8 MB 1.2 MB/s eta 0:00:22
   - -------------------------------------- 1.0/25.8 MB 1.2 MB/s eta 0:00:22
   - -------------------------------------- 1.0/25.8 MB 1.2 MB/s eta 0:00:22
   - -------------------------------------- 1.0/25.8 MB 1.2 MB/s eta 0:00:22
   -- ------------------------------------- 1.3/25.8 M

In [9]:
import gradio as gr
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
import torch
import soundfile as sf
import os

# 번역 모델 및 토크나이저
translation_model = M2M100ForConditionalGeneration.from_pretrained("facebook/m2m100_418M")
translation_tokenizer = M2M100Tokenizer.from_pretrained("facebook/m2m100_418M")

# 음성 합성 모델 (TTS)
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")
speaker_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embeddings = torch.tensor(speaker_dataset[7306]["xvector"]).unsqueeze(0)

# 언어 매핑
LANGUAGES = {
    "Korean": "ko",
    "English": "en",
    "Chinese": "zh",
    "Japanese": "ja",
    "French": "fr",
    "Spanish": "es",
    "German": "de",
    "Hindi": "hi"
}

# 번역 + 음성합성 함수
def translate_and_speak(text, src_lang_name, tgt_lang_name):
    if not text.strip():
        return "", None
    
    src_lang = LANGUAGES[src_lang_name]
    tgt_lang = LANGUAGES[tgt_lang_name]

    # 번역
    translation_tokenizer.src_lang = src_lang
    encoded = translation_tokenizer(text, return_tensors="pt")
    generated_tokens = translation_model.generate(
        **encoded, forced_bos_token_id=translation_tokenizer.get_lang_id(tgt_lang)
    )
    translated_text = translation_tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

    # 영어일 경우 음성 생성
    audio_path = None
    if tgt_lang == "en":
        tts_input = tts_processor(text=translated_text, return_tensors="pt")
        speech = tts_model.generate_speech(tts_input["input_ids"], speaker_embeddings, vocoder=vocoder)
        audio_path = "speech.wav"
        sf.write(audio_path, speech.numpy(), samplerate=16000)

    return translated_text, audio_path

# Gradio UI
with gr.Blocks() as demo:
    gr.Markdown("## 🌐 다국어 번역기 + 🎧 영어 음성 지원")

    with gr.Row():
        src_lang = gr.Dropdown(list(LANGUAGES.keys()), label="원본 언어", value="Korean")
        tgt_lang = gr.Dropdown(list(LANGUAGES.keys()), label="번역 언어", value="English")

    with gr.Row():
        input_text = gr.Textbox(lines=8, label="원본 텍스트")
        output_text = gr.Textbox(lines=8, label="번역 결과", interactive=False)

    audio_output = gr.Audio(label="영어 음성 (자동 생성)", type="filepath")

    translate_button = gr.Button("번역하기")

    translate_button.click(
        fn=translate_and_speak,
        inputs=[input_text, src_lang, tgt_lang],
        outputs=[output_text, audio_output]
    )

demo.launch()


C:\Users\haram4th\miniforge3\envs\ai_seving\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\haram4th\.cache\huggingface\hub\models--microsoft--speecht5_hifigan. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Error while downloading from https://cdn-lfs.hf.co/repos/67/8c/678ccd4f577bd6e95812006529803ced

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
